In [ ]:
#Checking basic features needed
import pandas as pd
import numpy as np

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

(2200, 8)
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label           object
dtype: object
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64
                 N            P            K  temperature     humidity  \
count  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
mean     50.551818    53.362727    48.149091    25.616244    71.481779   
std      36.917334    32.985883    50.647931     5.063749    22.263812   
min       0.000000     5.000000     5.000000     8.825675    14.258040   
25%      21.000000    28.000000    20.000000    22.769375    60.261953   
50%      37.000000    51.000000    32.000000    25.598693    80.473146   
75%      84.250000    68.000000    49.000000    28.561654    89.948771   
max     140.000000   145.000000   205.000000    43.

In [ ]:
# Check if features need scaling
print("Feature ranges:")
for col in features:
    print(f"{col:15} min: {df[col].min():.2f}  max: {df[col].max():.2f}")

Feature ranges:
N               min: 0.00  max: 140.00
P               min: 5.00  max: 145.00
K               min: 5.00  max: 205.00
temperature     min: 8.83  max: 43.68
humidity        min: 14.26  max: 99.98
ph              min: 3.50  max: 9.94
rainfall        min: 20.21  max: 298.56


In [ ]:
X = df[features]   # 7 features
y = df['Crop_Encoded']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Unique crops:   {y.nunique()}")

Features shape: (2200, 7)
Target shape:   (2200,)
Unique crops:   22


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {X_train.shape[0]}")  # 1760
print(f"Testing samples:  {X_test.shape[0]}")   # 440

# Train model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline.fit(X_train, y_train)
print("Training complete!")

Training samples: 1760
Testing samples:  440
Training complete!


In [ ]:
y_pred = pipeline.predict(X_test)

from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Accuracy: 99.55%


In [ ]:
import joblib

# Save pipeline (scaler + model together)
joblib.dump(pipeline, 'crop_recommendation_model.pkl')

# Save label encoder (needed to decode predictions later)
joblib.dump(le, 'label_encoder.pkl')

print("Model saved!")
print(f"Model size: {os.path.getsize('crop_recommendation_model.pkl') / (1024*1024):.2f} MB")

Model saved!
Model size: 3.41 MB


In [ ]:
import joblib
import numpy as np

model = joblib.load('crop_recommendation_model.pkl')
le = joblib.load('label_encoder.pkl')

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    sample = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    prediction = model.predict(sample)
    crop = le.inverse_transform(prediction)
    return crop[0]

# Example
print(recommend_crop(90, 42, 43, 20.8, 82.0, 6.5, 202.9))

rice


In [ ]:
#Crop count summary
crop_counts = df['label'].value_counts()
print(crop_counts.head(10))
print(f"\nTotal unique crops: {len(crop_counts)}")

label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
Name: count, dtype: int64

Total unique crops: 22
